# 让模型使用工具

模型只有同时具备工具调用（与环境交互获取实时信息）和结构化输出（以规范格式生成可解析、可执行的结果）这两项能力，
才能真正从“聊天机器”进化为“自主决策的执行体”。

## 工具注册

普通模型只能输出文本，但调用 `bind_tools` 后，模型能在回复中返回工具调用的命令，告诉程序它要使用哪个工具。

In [1]:
from typing import Optional

from pydantic import BaseModel, Field

from models import Models

model = Models.flash()

# 字典描述工具, 使用 OpenAI function calling 格式
# https://developers.openai.com/api/docs/guides/function-calling
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "parameters": {
                "type": "object",
                "description": "查询指定城市的天气",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# 使用 bind_tools 绑定工具到模型
model_with_tools = model.bind_tools(tools)

resp = model_with_tools.invoke("今天天津天气怎么样？")

if resp.tool_calls:
    print("模型想要调用:")
    for tc in resp.tool_calls:
        print(f"  工具:{tc['name']}")
        print(f"  参数:{tc['args']}")
        print(f"  ID:{tc['id']}")
else:
    print("模型不想调用工具")

模型想要调用:
  工具:get_weather
  参数:{'city': '天津'}
  ID:call_00_2t5PDv7zmI6deAGS8bvv9416


当前，大语言模型本身并不具备执行工具的能力，它所能做的，仅仅是在对话中生成一条结构化的工具调用指令（即 tool_calls），
告诉你“应该用哪个工具、传什么参数”。
至于如何解析这条指令、如何真正调用函数、如何处理返回结果——这些都需要开发者手动编写代码来实现。

这种“模型发令、代码执行”的分离设计，固然带来了高度的灵活性，但也带来了不小的重复劳动和工程成本。
所幸，LangChain 对这一模式进行了优雅的封装，
能够自动解析模型的 tool_calls 输出，并直接路由到对应的工具函数执行，将原本繁琐的胶水代码大幅简化。

当然，无论框架封装得多么智能，其底层逻辑始终不变：模型负责生成调用意图，开发者（或框架）负责执行实际动作。理解这一点，是掌握 Agent 设计范式的关键一步。

## 使用 `Pydantic` 描述工具

`Pydantic` 定义的参数更方便，不用手动写字典，手动写字典麻烦又容易出错。

In [2]:

# 使用 Pydantic 定义工具的输入参数结构
class WeatherInput(BaseModel):
    city: str = Field(description="城市名称，如：北京 天津")
    unit: str = Field(default="摄氏度", description="温度单位：摄氏度 or 华氏度")

model = Models.flash()

model_with_tools = model.bind_tools([WeatherInput])

resp = model_with_tools.invoke("查询今天天津天气怎么样")

resp

AIMessage(content='', additional_kwargs={'refusal': None, 'reasoning_content': "The user wants to query today's weather in Tianjin. Let me call the weather tool."}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 391, 'total_tokens': 454, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 19, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 384, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 7}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '0d40e6a8-bbb7-4ed3-9555-9168195bd6b9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a07a48-f277-7213-8c27-90e59168699c-0', tool_calls=[{'name': 'WeatherInput', 'args': {'city': '天津'}, 'id': 'call_00_nRNtVOoD

# 结构化输出

`with_structured_output()` 是在“格式化输出”场景下的工具调用的便捷封装，更偏向于模型本身的工具能力。
`with_structured_output` 解决的是“格式问题”，而工具调用解决的是“能力问题”。
两者是互补关系，不是替代关系。如果既要固定格式，又要执行操作，那就得配合着用。

In [3]:
class Answer(BaseModel):
    """用户问题答案、理由以及关联问题"""
    answer: str = Field(description="问题答案")
    justification: str = Field(description="理由")
    relation_question: Optional[str] = Field(default=None, description="关联问题")


model = Models.chat()
structured_model = model.with_structured_output(Answer) 

resp = structured_model.invoke("天文是工科还是理科？")
resp

Answer(answer='天文学属于理科（理学），具体来说是理学门类下的"天文学"一级学科。', justification='天文学主要以观测、理论研究为基础，通过物理学、数学等方法研究宇宙中的天体、星系以及宇宙本身的结构、起源和演化。在学科分类中，天文学在中国教育部学科体系中属于理学门类（代码07），是基础研究型学科，与物理学、化学、数学等并列，而非工科。虽然天文学在应用层面会涉及工程技术（如望远镜研制、航天探测等），但其学科本质属于自然科学（理科），强调对自然规律的理论探索和理解。', relation_question='天文专业毕业后就业前景如何？')

可以加入参数：`include_raw=True`，显示消息原始内容，解析结果，错误等。将是一个包含 'raw', 'parsed', 'parsing_error' 的字典

In [4]:

structured_model = Models.chat().with_structured_output(Answer, include_raw=True)
resp = structured_model.invoke("天文是工科还是理科？")
resp

{'raw': AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 266, 'prompt_tokens': 350, 'total_tokens': 616, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 94}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '378b2ad2-1f67-40c5-a833-efab0f723741', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a07a48-fee1-7540-a20b-884e1b28a09f-0', tool_calls=[{'name': 'Answer', 'args': {'answer': '天文学属于理科（理学），不是工科。', 'justification': '天文学（Astronomy）在中国高等教育学科体系中属于理学门类（代码07），是基础科学的一个分支，与物理学、化学、生物学等并列。天文学主要研究宇宙中天体的运动、结构、性质、演化规律以及宇宙的起源和演化，属于探索自然规律的基础研究。\n\n根据我国教育部学科分类，天文学是一级学科，下设天体物理、天体测量与天体力学等二级学科。在本科招生专业目录中，天文学专业（070401）属于理学门类，毕业时授予理学学士学位。\n\n虽然天文学在实际研究中需要大量应

# 消息类型

## 四种核心消息类型
LangChain 定义了四种核心的消息类型，分别在对话中担任不同的角色。

| 类型           | 角色 | 说明           | 示例               |
|----------------|------|----------------|--------------------|
| `HumanMessage` | 用户 | 用户发送的消息 | “今天天气怎么样？” |
| `AIMessage`    | AI 助手 | 模型的回复，可能包含 tool_calls | “今天天津晴天，21摄氏度。” |
| `SystemMessage` | 系统 | 系统提示词，定义 AI 的角色和行为规则 | “你是一个专业的天气助手：” |
| `ToolMessage` | 工具 | 工具执行后的返回结果 | “天津：晴，21摄氏度” |

### `HumanMessage` 用户消息

In [5]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage


human_msg = HumanMessage("你好，你是？")

human_msg

HumanMessage(content='你好，你是？', additional_kwargs={}, response_metadata={})

### SystemMessage 系统消息

系统消息可以用于设定 ai 的行为，角色等，类似于提示词。

In [6]:
system_msg = SystemMessage("你是个话少的人，请尽量一句话回复。")

system_msg

SystemMessage(content='你是个话少的人，请尽量一句话回复。', additional_kwargs={}, response_metadata={})

### AIMessage 模型返回消息

AI 返回的消息，可能包换工具调用、附加token使用量等数据。
它除了是 ai 的返回值信息，也可以作为模型的输入标识这个是 ai 的输出，模型是无状态的，所有的对话都是携带了之前所有的历史记录，比如系统消息，用户消息，ai消息，工具调用请求，工具调用结果。

In [7]:
msgs = [system_msg, human_msg]

ai_message = model.invoke(msgs)

ai_message

AIMessage(content='我是话少，但愿意倾听的伙伴。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 19, 'total_tokens': 29, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 19}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '91383134-ab29-4faf-b0e4-0658a12413f1', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a07a49-0a42-7171-b739-dd120860c5b1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 10, 'total_tokens': 29, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})

如果使用了工具调用，AIMessage 中还会包含工具调用请求。

In [8]:
msgs = [
    system_msg,
    HumanMessage("天津今天天气如何？")
]

ai_message = model_with_tools.invoke(msgs)
ai_message.tool_calls

[{'name': 'WeatherInput',
  'args': {'city': '天津'},
  'id': 'call_00_aV6yhguRJwNVGXP9WMTL5098',
  'type': 'tool_call'}]

AI 消息的附加信息，比如响应信息、token用量。

In [9]:
ai_message.usage_metadata

{'input_tokens': 402,
 'output_tokens': 52,
 'total_tokens': 454,
 'input_token_details': {'cache_read': 384},
 'output_token_details': {'reasoning': 8}}

In [10]:
ai_message.response_metadata

{'token_usage': {'completion_tokens': 52,
  'prompt_tokens': 402,
  'total_tokens': 454,
  'completion_tokens_details': {'accepted_prediction_tokens': None,
   'audio_tokens': None,
   'reasoning_tokens': 8,
   'rejected_prediction_tokens': None,
   'text_tokens': None},
  'prompt_tokens_details': {'audio_tokens': None,
   'cache_write_tokens': None,
   'cached_tokens': 384,
   'image_tokens': None,
   'text_tokens': None},
  'prompt_cache_hit_tokens': 384,
  'prompt_cache_miss_tokens': 18},
 'model_provider': 'deepseek',
 'model_name': 'deepseek-v4-flash',
 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
 'id': '05321ea2-6072-4a73-9b9a-7aa38e8c5825',
 'finish_reason': 'tool_calls',
 'logprobs': None}

### ToolMessage 工具返回结果

当 ai 返回工具调用请求后，可以使用 ToolMessage 包裹工具的调用结果给 ai，让 ai 理解工具调用的返回结果。

In [11]:
# 这里是模拟了用户消息，ai 返回工具调用请求，工具调用结果，然后让 ai 对结果进行总结。
msgs = [
    system_msg,
    HumanMessage("今天天津天气怎么样"),
    AIMessage(content="", tool_calls=[
        {
            "name": "get_weather",
            "args": {"city": "天津"},
            "id": "call_get_weather",
            "type": "tool_call"
        }
    ]),
    ToolMessage(
        content="阴天，18度，湿度 80%",
        tool_call_id="call_get_weather", # 要和工具调用请求的 id 一致, 如果不一致可能会报错
        name="get_weather" # 工具名称
    )
]
resp = model.invoke(msgs)
resp

AIMessage(content='天津阴天，18度，湿度大，带伞。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 85, 'total_tokens': 98, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 85}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '8a33b0f3-9b9f-4885-9dcf-3057d57ae3c6', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a07a49-1264-79a1-a1da-e03da8e6ec76-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 85, 'output_tokens': 13, 'total_tokens': 98, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})

### AIMessageChunk 流式输出的消息片段

In [12]:
for chunk in model.stream("背一下逍遥游"):
    print(chunk.content, end="", flush=True)
# 执行能看到，流式的打印

《逍遥游》是庄子代表作，这里为您背诵其中最为经典的段落（从开头至“至人无己”部分，含全文核心思想）：

---

**北冥有鱼，其名为鲲。鲲之大，不知其几千里也。化而为鸟，其名为鹏。鹏之背，不知其几千里也；怒而飞，其翼若垂天之云。是鸟也，海运则将徙于南冥。南冥者，天池也。**

**《齐谐》者，志怪者也。《谐》之言曰：“鹏之徙于南冥也，水击三千里，抟扶摇而上者九万里，去以六月息者也。”野马也，尘埃也，生物之以息相吹也。天之苍苍，其正色邪？其远而无所至极邪？其视下也，亦若是则已矣。**

**且夫水之积也不厚，则其负大舟也无力。覆杯水于坳堂之上，则芥为之舟；置杯焉则胶，水浅而舟大也。风之积也不厚，则其负大翼也无力。故九万里，则风斯在下矣，而后乃今培风；背负青天而莫之夭阏者，而后乃今将图南。**

**蜩与学鸠笑之曰：“我决起而飞，抢榆枋而止，时则不至，而控于地而已矣，奚以之九万里而南为？”适莽苍者，三餐而反，腹犹果然；适百里者，宿舂粮；适千里者，三月聚粮。之二虫又何知！**

**小知不及大知，小年不及大年。奚以知其然也？朝菌不知晦朔，蟪蛄不知春秋，此小年也。楚之南有冥灵者，以五百岁为春，五百岁为秋；上古有大椿者，以八千岁为春，八千岁为秋，此大年也。而彭祖乃今以久特闻，众人匹之，不亦悲乎！**

**汤之问棘也是已。穷发之北，有冥海者，天池也。有鱼焉，其广数千里，未有知其修者，其名为鲲。有鸟焉，其名为鹏，背若泰山，翼若垂天之云，抟扶摇羊角而上者九万里，绝云气，负青天，然后图南，且适南冥也。斥鴳笑之曰：“彼且奚适也？我腾跃而上，不过数仞而下，翱翔蓬蒿之间，此亦飞之至也。而彼且奚适也？”此小大之辩也。**

**故夫知效一官，行比一乡，德合一君，而征一国者，其自视也，亦若此矣。而宋荣子犹然笑之。且举世誉之而不加劝，举世非之而不加沮，定乎内外之分，辩乎荣辱之境，斯已矣。彼其于世，未数数然也。虽然，犹有未树也。夫列子御风而行，泠然善也，旬有五日而后反。彼于致福者，未数数然也。此虽免乎行，犹有所待者也。**

**若夫乘天地之正，而御六气之辩，以游无穷者，彼且恶乎待哉？故曰：至人无己，神人无功，圣人无名。**

---

此段为《逍遥游》主体，囊括了鲲鹏寓言、小大之辩、无待逍遥三重境界。若需要全文或逐句翻译，可随时告诉我。

## ContentBlock 结构化消息内容

如果你的消息包含文本和图片，可以使用 ContentBlock。

In [17]:
from langchain.messages import TextContentBlock, ImageContentBlock

msg = HumanMessage(content=[
    TextContentBlock(type="text", text="看一下图片内容"),
    ImageContentBlock(
        type="image",
        url="https://picsum.photos/200/300.jpg"
    )
])

resp = model.invoke([msg])
resp # deepseek 模型不支持图片。

AIMessage(content='我无法直接查看或读取图片内容。\n\n不过，如果你愿意的话，可以把图片中的文字、题目或你遇到的问题**打字描述**出来，我会很乐意帮你分析或解答。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 13, 'total_tokens': 51, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 13}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '1fb8b5b6-bf76-4d58-ace5-f42071f1335b', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a07a4e-eaf5-72e3-95a7-1e6ae236b1ec-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 38, 'total_tokens': 51, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})